# Compute Sentence-Transformer Embeddings

Reads the **iGEM teams** and **SynBio OpenAlex** datasets, concatenates title + abstract
for each record, computes embeddings with a sentence-transformer model, and saves the
results as NumPy arrays in `../assets/embeddings/`.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
MODEL_NAME = 'all-MiniLM-L6-v2'

ASSETS_DIR     = os.path.join('..', 'assets')
EMBEDDINGS_DIR = os.path.join(ASSETS_DIR, 'embeddings')
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)

## 1. Load datasets

In [ ]:
# iGEM teams dataset (columns: TI, AB)
teams = pd.read_csv(os.path.join(ASSETS_DIR, 'igem.txt'), sep='\t')
print(f'Teams: {len(teams):,} rows')

# SynBio OpenAlex dataset (columns: title, abstract)
papers = pd.read_csv(os.path.join(ASSETS_DIR, 'synbio_openalex.txt'), sep='\t')
print(f'Papers: {len(papers):,} rows')

## 2. Prepare text for embedding

Concatenate title and abstract into a single `text` field, clean minimally
(remove non-alphabetic characters, collapse whitespace), and drop rows with no text.

In [ ]:
def prepare_text(df: pd.DataFrame, title_col: str, abstract_col: str) -> pd.DataFrame:
    """Concatenate title + abstract, clean, and drop empty rows."""
    df = df.copy()
    df[title_col] = df[title_col].fillna('').astype(str)
    df[abstract_col] = df[abstract_col].fillna('').astype(str)
    df['text'] = (df[title_col] + ' ' + df[abstract_col]).str.strip()
    # Light cleaning: remove non-alpha chars, collapse whitespace
    df['text'] = df['text'].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', x))
    df['text'] = df['text'].str.strip()
    df = df[df['text'] != ''].reset_index(drop=True)
    return df

In [ ]:
teams_corpus  = prepare_text(teams,  title_col='TI', abstract_col='AB')
papers_corpus = prepare_text(papers, title_col='title', abstract_col='abstract')

print(f'Teams corpus:  {len(teams_corpus):,}')
print(f'Papers corpus: {len(papers_corpus):,}')

## 3. Compute embeddings

In [ ]:
model = SentenceTransformer(MODEL_NAME)
print(f'Loaded model: {MODEL_NAME}')

In [ ]:
print('Encoding teams corpus...')
teams_embeddings = model.encode(teams_corpus['text'].tolist(), show_progress_bar=True)
print(f'  shape: {teams_embeddings.shape}')

In [ ]:
print('Encoding papers corpus...')
papers_embeddings = model.encode(papers_corpus['text'].tolist(), show_progress_bar=True)
print(f'  shape: {papers_embeddings.shape}')

## 4. Save embeddings and corpus metadata

In [ ]:
# Save embeddings as .npy
np.save(os.path.join(EMBEDDINGS_DIR, 'teams_embeddings.npy'), teams_embeddings)
np.save(os.path.join(EMBEDDINGS_DIR, 'papers_embeddings.npy'), papers_embeddings)

# Save corpus text + IDs so downstream notebooks can align rows
teams_corpus[['UT', 'text']].to_csv(
    os.path.join(EMBEDDINGS_DIR, 'teams_corpus.txt'), sep='\t', index=False
)
papers_corpus[['id', 'text']].to_csv(
    os.path.join(EMBEDDINGS_DIR, 'papers_corpus.txt'), sep='\t', index=False
)

print('Saved to', EMBEDDINGS_DIR)
for f in sorted(os.listdir(EMBEDDINGS_DIR)):
    size_mb = os.path.getsize(os.path.join(EMBEDDINGS_DIR, f)) / 1e6
    print(f'  {f:40s} {size_mb:>7.1f} MB')